# List Local Units by Country

**Environment:** Production (`goadmin.ifrc.org`)

This notebook fetches all local units for a given country from the IFRC GO API.
You can filter by country ISO3 code, unit type, and status.

---

## 1. Setup & Authentication

Your token will **not** be saved in the notebook output.

In [9]:
import requests
import json
import getpass
import os

# ============================================================
# ENVIRONMENT - This notebook uses PRODUCTION
# ============================================================
BASE_URL = "https://goadmin.ifrc.org/api/v2"

# Securely input your API token (will not be stored in notebook output)
TOKEN = getpass.getpass("Enter your IFRC GO API token: ")

HEADERS = {
    "Authorization": f"Token {TOKEN}",
    "Accept": "application/json"
}

print("Authentication configured (Production)")

Authentication configured (Production)


## 2. Configuration

Set the country and optional filters below.

**Common ISO3 codes:** DEU (Germany), CHE (Switzerland), FRA (France), GBR (UK), ESP (Spain), NLD (Netherlands)

In [10]:
# ============================================================
# CONFIGURATION - Change these values to target different data
# ============================================================

COUNTRY_ISO3 = "DEU"      # Country ISO3 code (e.g., DEU, CHE, FRA, GBR)
LIMIT = 50                 # Records per page (max typically 50)

# Optional filters (set to None to skip)
UNIT_TYPE = None           # e.g., 1 = Branch, 2 = Health Care
UNIT_STATUS = None         # e.g., 1 = Validated, 2 = Pending, 4 = Externally Managed

print(f"Country:          {COUNTRY_ISO3}")
print(f"Records per page: {LIMIT}")
print(f"Type filter:      {UNIT_TYPE or 'All'}")
print(f"Status filter:    {UNIT_STATUS or 'All'}")

Country:          DEU
Records per page: 50
Type filter:      All
Status filter:    All


## 3. Fetch Local Units (with Pagination)

In [11]:
def fetch_local_units(country_iso3, limit=50, unit_type=None, unit_status=None):
    """
    Fetch all local units for a country with pagination.
    
    Args:
        country_iso3: ISO3 country code (e.g., 'DEU')
        limit: Records per page
        unit_type: Optional type filter (1=Branch, 2=Health Care, etc.)
        unit_status: Optional status filter (1=Validated, 2=Pending, etc.)
    
    Returns:
        List of local unit records
    """
    all_records = []
    offset = 0
    
    print(f"Fetching local units for {country_iso3}...\n")
    
    while True:
        params = {
            "country__iso3": country_iso3,
            "limit": limit,
            "offset": offset
        }
        
        # Add optional filters
        if unit_type is not None:
            params["type"] = unit_type
        if unit_status is not None:
            params["status"] = unit_status
        
        try:
            response = requests.get(f"{BASE_URL}/local-units/", headers=HEADERS, params=params)
            response.raise_for_status()
            data = response.json()
            
            # Handle paginated response
            if isinstance(data, dict):
                records = data.get("results", [])
                total_count = data.get("count", len(records))
            elif isinstance(data, list):
                records = data
                total_count = len(records)
            else:
                print(f"Unexpected response format: {type(data)}")
                break
            
            if not records:
                break
            
            all_records.extend(records)
            print(f"  Fetched {len(records)} records (offset={offset}). Total so far: {len(all_records)}/{total_count}")
            
            # Check if done
            if len(all_records) >= total_count or len(records) < limit:
                break
            
            offset += limit
            
        except requests.exceptions.HTTPError as e:
            print(f"HTTP Error: {e}")
            print(f"   Response: {response.text[:500]}")
            break
        except requests.exceptions.RequestException as e:
            print(f"Request Error: {e}")
            break
    
    print(f"\nDone. Fetched {len(all_records)} total records.")
    return all_records


# Fetch the data
local_units = fetch_local_units(COUNTRY_ISO3, LIMIT, UNIT_TYPE, UNIT_STATUS)

Fetching local units for DEU...

  Fetched 50 records (offset=0). Total so far: 50/541
  Fetched 50 records (offset=50). Total so far: 100/541
  Fetched 50 records (offset=100). Total so far: 150/541
  Fetched 50 records (offset=150). Total so far: 200/541
  Fetched 50 records (offset=200). Total so far: 250/541
  Fetched 50 records (offset=250). Total so far: 300/541
  Fetched 50 records (offset=300). Total so far: 350/541
  Fetched 50 records (offset=350). Total so far: 400/541
  Fetched 50 records (offset=400). Total so far: 450/541
  Fetched 50 records (offset=450). Total so far: 500/541
  Fetched 41 records (offset=500). Total so far: 541/541

Done. Fetched 541 total records.


## 4. Explore the Data

In [12]:
if local_units:
    print(f"Summary for {COUNTRY_ISO3}")
    print(f"{'='*50}")
    print(f"Total units: {len(local_units)}")
    
    # Count by type
    type_counts = {}
    status_counts = {}
    for unit in local_units:
        # Type
        type_name = unit.get("type_details", {}).get("name", "Unknown")
        type_counts[type_name] = type_counts.get(type_name, 0) + 1
        # Status
        status_name = unit.get("status_details", "Unknown")
        status_counts[status_name] = status_counts.get(status_name, 0) + 1
    
    print(f"\nBy Type:")
    for t, count in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"   {t}: {count}")
    
    print(f"\nBy Status:")
    for s, count in sorted(status_counts.items(), key=lambda x: -x[1]):
        print(f"   {s}: {count}")
    
    # Show first record as sample
    print(f"\nSample record (first unit):")
    print(json.dumps(local_units[0], indent=2, ensure_ascii=False)[:1000])
else:
    print("No data fetched.")

Summary for DEU
Total units: 541

By Type:
   Health Care: 541

By Status:
   Externally Managed: 541

Sample record (first unit):
{
  "id": 28796,
  "country": 72,
  "local_branch_name": "Zentrale MH",
  "english_branch_name": "",
  "location_geojson": {
    "type": "Point",
    "coordinates": [
      6.577663,
      51.314146
    ]
  },
  "type": 2,
  "status": 4,
  "status_details": "Externally Managed",
  "address_loc": "Jakob-Lintzen-Straße 3, 47807 Krefeld",
  "address_en": "",
  "country_details": {
    "name": "Germany",
    "iso3": "DEU",
    "id": 72
  },
  "type_details": {
    "name": "Health Care",
    "code": 2,
    "id": 2,
    "colour": "#011e41",
    "image_url": "https://goadmin.ifrc.org/static/images/local_units/local_unit_type/Healthcare.png"
  },
  "health": 7855317,
  "health_details": {
    "id": 7855317,
    "health_facility_type": 9,
    "health_facility_type_details": {
      "id": 9,
      "image_url": "https://goadmin.ifrc.org/static/images/local_units/healt

## 5. Save to File

In [13]:
if local_units:
    # Create output directory if it doesn't exist
    os.makedirs("output", exist_ok=True)
    
    filename = f"output/{COUNTRY_ISO3.lower()}_local_units.json"
    
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(local_units, f, indent=2, ensure_ascii=False)
    
    print(f"Saved {len(local_units)} records to: {filename}")
else:
    print("No data to save.")

Saved 541 records to: output/deu_local_units.json
